# 1. Contrôles automatiques (sans IA)

Contrôles **déterministes et reproductibles** sur les exigences et l'AMDEC :
ASIL, décomposition ISO 26262-9, traçabilité, formulation, cotations, priorité d'action AIAG-VDA.

C'est la base fiable de l'outil : l'IA (notebooks suivants) vient en complément.

In [1]:
# Rend le package safety_agent importable depuis le dossier notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"

import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from safety_agent.loaders import load_project

# Remplacez par vos fichiers (.csv ou .xlsx)
REQS = EXAMPLES / "bms_exigences.csv"
FMEA = EXAMPLES / "bms_amdec.csv"

project, load_findings = load_project(REQS, FMEA)
print(f"{len(project.requirements)} exigences, {len(project.fmea)} lignes AMDEC")

11 exigences, 6 lignes AMDEC


## Les données

In [3]:
from safety_agent.notebook import requirements_df, fmea_df, findings_df, matrix_df
requirements_df(project)

,ID,Niveau,ASIL,Texte,Parents,Vérification,Statut
0,SG-01,SG,ASIL C,Le système doit empêcher la surcharge des cellules pouvant conduire à un emballement thermique.,,,Validé
1,SG-02,SG,ASIL B,Le système doit empêcher l'ouverture intempestive des contacteurs principaux en roulage.,,,Validé
2,FSR-01,FSR,ASIL C,Le BMS doit mesurer la tension de chaque cellule avec une précision de ±5 mV.,SG-01,Test,Validé
3,FSR-02,FSR,ASIL C,"Le BMS doit ouvrir les contacteurs principaux dans un délai de 100 ms lorsqu'une tension cellule dépasse 4,25 V.",SG-01,Test HIL,Validé
4,FSR-03,FSR,ASIL B(C),"Le microcontrôleur principal doit détecter une surtension cellule supérieure à 4,25 V.",SG-01,Test,En revue
5,FSR-04,FSR,ASIL A(C),"Un circuit de surveillance analogique indépendant doit détecter une surtension cellule supérieure à 4,30 V.",SG-01,Test,En revue
6,FSR-05,FSR,ASIL A,Le BMS doit maintenir les contacteurs fermés tant qu'aucun défaut critique n'est confirmé.,SG-02,Test,Brouillon
7,FSR-06,FSR,ASIL B,Le BMS doit réagir rapidement et de manière appropriée en cas de défaut de communication CAN et doit enregistrer un ...,SG-02,,Brouillon
8,TSR-01,TSR,ASIL C,Le logiciel doit commander l'ouverture des contacteurs via deux sorties matérielles indépendantes.,FSR-02,Revue + test,En revue
9,TSR-02,TSR,ASIL C,Le logiciel doit vérifier la cohérence des mesures de tension entre les deux chaînes d'acquisition.,FSR-99,Test,Brouillon


In [4]:
fmea_df(project)

,ID,Élément,Mode,Effet,Cause,S,O,D,AP,RPN,Actions,Exigences
0,FM-01,Capteur tension cellule,Dérive de la mesure (sous-estimation),"Surcharge non détectée, risque d'emballement thermique",Dérive de la référence de tension de l'ADC,10,4.0,5,H,200.0,,"FSR-01, FSR-03"
1,FM-02,Contacteur principal,Contacteur collé fermé,Impossible d'isoler la batterie en cas de défaut,Soudure des contacts sous fort courant,9,3.0,4,L,108.0,Évaluer un contacteur redondant,TSR-01
2,FM-03,Contacteur principal,Ouverture intempestive,Perte de propulsion en roulage,Perte d'alimentation de la bobine,8,4.0,6,M,192.0,,
3,FM-04,Communication CAN,Perte de trame,Le VCU ne reçoit plus l'état de charge ni les alarmes,,9,5.0,3,H,135.0,,FSR-77
4,FM-05,Watchdog externe,Watchdog inactif,Blocage logiciel non détecté,Mauvaise configuration à l'initialisation,7,2.0,9,M,126.0,Ajouter un test de démarrage du watchdog,SW-01
5,FM-06,Capteur température,Valeur figée,Surchauffe non détectée,Court-circuit du capteur NTC,10,NaN,6,NaN,NaN,,


## Lancer les contrôles

In [5]:
from safety_agent.checks import run_all_checks

findings = load_findings + run_all_checks(project)
df = findings_df(findings)
df["Niveau"].value_counts()

Niveau
error      9
warning    7
info       5
Name: count, dtype: int64

In [6]:
# Erreurs bloquantes
df[df["Niveau"] == "error"]

,Code,Niveau,Élément,Constat,Source
0,FMEA-001,error,FM-06,Cotation O manquante.,règle
1,FMEA-010,error,FM-01,Priorité d'action H sans action définie (action ou justification obligatoire).,règle
2,FMEA-010,error,FM-04,Priorité d'action H sans action définie (action ou justification obligatoire).,règle
3,FMEA-020,error,FM-04,Exigence(s) liée(s) inconnue(s) : FSR-77.,règle
4,FMEA-021,error,FM-04,Effet critique pour la sécurité (S≥9) sans exigence de sécurité tracée.,règle
5,FMEA-021,error,FM-06,Effet critique pour la sécurité (S≥9) sans exigence de sécurité tracée.,règle
6,REQ-012,error,SW-01,Exigence orpheline : aucun parent (rupture de traçabilité vers un objectif de sécurité).,règle
7,REQ-013,error,TSR-02,Parent inconnu : FSR-99.,règle
8,REQ-031,error,FSR-05,ASIL A inférieur à l'ASIL du parent SG-02 (ASIL B) sans décomposition déclarée.,règle


In [7]:
# Avertissements
df[df["Niveau"] == "warning"]

,Code,Niveau,Élément,Constat,Source
9,FMEA-004,warning,FM-04,Cause de la défaillance non renseignée.,règle
10,FMEA-011,warning,FM-03,Priorité d'action M sans action ni justification.,règle
11,REQ-003,warning,FSR-06,Méthode de vérification non renseignée (ISO 26262-8 §9).,règle
12,REQ-021,warning,FSR-06,"Termes ambigus ou non vérifiables : appropriée, rapidement.",règle
13,REQ-023,warning,FSR-06,"Exigence non atomique : 2 verbes d'obligation, à scinder.",règle
14,REQ-024,warning,FSR-06,Contrainte temporelle sans valeur chiffrée (FTTI / temps de réaction ?).,règle
15,TRA-001,warning,SG-02,Objectif de sécurité non couvert par l'AMDEC (aucun mode de défaillance relié).,règle


## Matrice de traçabilité
Objectif de sécurité → exigences dérivées → lignes AMDEC.

In [8]:
matrix_df(project)

,Objectif de sécurité,ASIL,Exigences,Lignes AMDEC
0,SG-01,ASIL C,"FSR-01, FSR-02, FSR-03, FSR-04, TSR-01","FM-01, FM-02"
1,SG-02,ASIL B,"FSR-05, FSR-06",AUCUNE


## Priorité d'action AIAG-VDA
Calcul pour n'importe quelle cotation S / O / D.

In [9]:
from safety_agent.checks.action_priority import action_priority
for s, o, d in [(10, 4, 5), (9, 2, 3), (7, 4, 7), (5, 2, 2)]:
    print(f"S={s} O={o} D={d} -> AP = {action_priority(s, o, d)}")

S=10 O=4 D=5 -> AP = H
S=9 O=2 D=3 -> AP = L
S=7 O=4 D=7 -> AP = H
S=5 O=2 D=2 -> AP = L


In [10]:
# Répartition des lignes AMDEC par priorité
fmea_df(project)["AP"].value_counts(dropna=False)

AP
H      2
M      2
L      1
NaN    1
Name: count, dtype: int64

## Exporter le rapport

In [11]:
from safety_agent.report import markdown_report

out = ROOT / "rapport_controles.md"
out.write_text(markdown_report(project, findings), encoding="utf-8")
print("Rapport écrit :", out.relative_to(ROOT))

Rapport écrit : rapport_controles.md
